# **Variasi Bahasa Lokal — Revisi Aturan dan Dataset Kandidat**

---

**Project:** KP Agentic AI + LLM  
**Tahap:** Step 10–11  
**Dataset:** MASSIVE-Agents `id-ID` development

Aturan variasi diperbaiki dari hasil tinjauan pilot 40 instruksi. Lima kategori tetap dipakai: slang/informal, singkatan, typo, campuran ID–EN, dan ID–Jawa. Seluruh hasil masih berupa kandidat dan belum dipakai untuk evaluasi akhir model.

## **1. Dataset dan Konfigurasi**

---

Dataset berasal dari split `development` yang telah digunakan pada baseline. Pembuatan variasi tidak mengubah data asli, ground truth, fungsi target, atau `group_id`. Pengujian ini tidak memakai data test.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import os
import random
import re
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

SEED = 20261110
TARGET_AUDIT_PER_KATEGORI = 100
KATEGORI = ["slang", "singkatan", "typo", "code_switch_id_en", "code_switch_id_jawa"]
ROOT_OVERRIDE = None


def cari_root():
    kandidat = []
    if ROOT_OVERRIDE:
        kandidat.append(Path(ROOT_OVERRIDE))
    if os.environ.get("KP_AGENTIC_AI_ROOT"):
        kandidat.append(Path(os.environ["KP_AGENTIC_AI_ROOT"]))
    cwd = Path.cwd().resolve()
    kandidat += [cwd, *cwd.parents, Path(r"D:\KP-Agentic-AI")]
    for root in kandidat:
        p = root / "data" / "processed" / "massive_agents_reviewed" / "id-ID_development.jsonl"
        if p.is_file():
            return root
    raise FileNotFoundError("Dataset id-ID_development.jsonl tidak ditemukan pada folder project.")

ROOT = cari_root()
INPUT = ROOT / "data" / "processed" / "massive_agents_reviewed" / "id-ID_development.jsonl"
OUTPUT = ROOT / "results" / "variasi_bahasa_terkontrol" / "development_v2"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Dataset:", INPUT)
print("Output:", OUTPUT)

Dataset: D:\KP-Agentic-AI\data\processed\massive_agents_reviewed\id-ID_development.jsonl
Output: D:\KP-Agentic-AI\results\variasi_bahasa_terkontrol\development_v2


## **2. Catatan Perbaikan dari Pilot**

---

Tinjauan pilot menemukan beberapa masalah: penggantian kata yang tidak alami, ragam informal yang belum tentu slang leksikal, dan instruksi induk yang perlu dicek lagi. Pada versi ini, penggantian `waktu → time`, `apa → opo`, dan `berapa → pira` secara bebas tidak digunakan. Campuran bahasa menggunakan frasa yang lebih terbatas. Typo tetap dibuat dengan satu perubahan karakter secara rule-based.

Status kelayakan tetap membutuhkan validasi manusia. Catatan pilot berbantuan AI tidak diperlakukan sebagai keputusan final.

## **3. Seleksi Instruksi Induk**

---

Instruksi formal dipilih menggunakan penyaringan otomatis yang sama dengan pilot. Satu instruksi dipilih untuk tiap `group_id`. Status formal dan kesesuaian GT belum dianggap tervalidasi.

In [2]:
with INPUT.open("r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

assert len(data) == 397, "Jumlah data Indonesia development berubah."
assert len({r["row_key"] for r in data}) == len(data)
assert all(r["split"] == "development" and r["language_code"] == "id-ID" for r in data)

KATA_INFORMAL_ASAL = re.compile(
    r"\b(olly|siri|alexa|hey|ayo|aku|gak|nggak|ngga|pengen|dong|ngelawak|gue|gw|lo|lu)\b", re.I
)

def normal(teks):
    return " ".join(teks.casefold().split())

formal_awal = [r for r in data if r["question"].strip() and not KATA_INFORMAL_ASAL.search(r["question"])]
induk = []
group_seen = set()
teks_seen = set()
for row in sorted(formal_awal, key=lambda x: x["row_key"]):
    teks = normal(row["question"])
    if row["group_id"] in group_seen or teks in teks_seen:
        continue
    group_seen.add(row["group_id"])
    teks_seen.add(teks)
    induk.append(row)

print("ID development:", len(data))
print("Kandidat formal otomatis:", len(formal_awal))
print("Induk unik (group_id dan teks):", len(induk))

ID development: 397
Kandidat formal otomatis: 365
Induk unik (group_id dan teks): 359


## **4. Aturan Variasi Bahasa**

---

Slang dan singkatan memakai daftar padanan terbatas. Campuran EN dan Jawa memakai frasa yang lazim dalam perintah singkat. Kata atau frasa yang merupakan isi argumen GT tidak diganti. Satu varian dibuat untuk setiap kategori yang mempunyai aturan cocok.

Pada kategori `slang`, perubahan kata ganti seperti `saya → aku` diberi penanda `ragam_informal`. Hal ini tidak dianggap otomatis sebagai slang leksikal.

In [3]:
POLA = {
    "slang": [
        (r"(?<!\w)bagaimana(?!\w)", "gimana", "slang_leksikal", "tinggi"),
        (r"(?<!\w)bisakah(?!\w)", "bisa nggak", "slang_leksikal", "sedang"),
        (r"(?<!\w)ingin(?!\w)", "pengen", "slang_leksikal", "tinggi"),
        (r"(?<!\w)tidak(?!\w)", "nggak", "slang_leksikal", "tinggi"),
        (r"(?<!\w)beritahu(?!\w)", "kasih tahu", "slang_leksikal", "tinggi"),
        (r"(?<!\w)beri tahu(?!\w)", "kasih tahu", "slang_leksikal", "tinggi"),
        (r"^tolong\b", "tolong dong", "partikel_informal", "sedang"),
        (r"(?<!\w)kamu(?!\w)", "lu", "slang_leksikal", "sedang"),
        (r"(?<!\w)saya(?!\w)", "aku", "ragam_informal", "sedang"),
    ],
    "singkatan": [
        (r"(?<!\w)bagaimana(?!\w)", "gmn", "singkatan_kamus", "tinggi"),
        (r"(?<!\w)sekarang(?!\w)", "skrg", "singkatan_kamus", "tinggi"),
        (r"(?<!\w)tolong(?!\w)", "tlg", "singkatan_kamus", "tinggi"),
        (r"(?<!\w)berapa(?!\w)", "brp", "singkatan_kamus", "tinggi"),
        (r"(?<!\w)apakah(?!\w)", "apkh", "singkatan_kamus", "sedang"),
        (r"(?<!\w)dengan(?!\w)", "dgn", "singkatan_kamus", "tinggi"),
        (r"(?<!\w)untuk(?!\w)", "utk", "singkatan_kamus", "tinggi"),
        (r"(?<!\w)sudah(?!\w)", "sdh", "singkatan_kamus", "tinggi"),
        (r"(?<!\w)yang(?!\w)", "yg", "singkatan_kamus", "tinggi"),
        (r"(?<!\w)jangan(?!\w)", "jgn", "singkatan_kamus", "tinggi"),
        (r"(?<!\w)saya(?!\w)", "sy", "singkatan_kamus", "sedang"),
    ],
    "code_switch_id_en": [
        (r"^tolong\b", "please", "kata_perintah", "tinggi"),
        (r"(?<!\w)cek email(?!\w)", "check email", "frasa_perintah", "tinggi"),
        (r"(?<!\w)putar lagu(?!\w)", "play lagu", "frasa_perintah", "tinggi"),
        (r"(?<!\w)putar musik(?!\w)", "play musik", "frasa_perintah", "tinggi"),
        (r"(?<!\w)mainkan lagu(?!\w)", "play lagu", "frasa_perintah", "tinggi"),
        (r"(?<!\w)mainkan musik(?!\w)", "play musik", "frasa_perintah", "tinggi"),
        (r"(?<!\w)matikan lampu(?!\w)", "turn off lampu", "frasa_perintah", "tinggi"),
        (r"(?<!\w)nyalakan lampu(?!\w)", "turn on lampu", "frasa_perintah", "tinggi"),
        (r"(?<!\w)atur alarm(?!\w)", "set alarm", "frasa_perintah", "tinggi"),
        (r"(?<!\w)hapus alarm(?!\w)", "delete alarm", "frasa_perintah", "sedang"),
    ],
    "code_switch_id_jawa": [
        (r"^tolong\b", "tulung", "jawa_ngoko", "sedang"),
        (r"(?<!\w)bagaimana cara(?!\w)", "piye carane", "jawa_ngoko", "sedang"),
        (r"(?<!\w)jam berapa(?!\w)", "jam piro", "jawa_ngoko", "sedang"),
        (r"(?<!\w)berapa harga(?!\w)", "piro regane", "jawa_ngoko", "sedang"),
        (r"(?<!\w)berapa lama(?!\w)", "pira suwene", "jawa_ngoko", "sedang"),
        (r"(?<!\w)di mana(?!\w)", "neng endi", "jawa_ngoko", "sedang"),
        (r"(?<!\w)untuk saya(?!\w)", "kanggo aku", "jawa_ngoko", "sedang"),
        (r"(?<!\w)dengan saya(?!\w)", "karo aku", "jawa_ngoko", "sedang"),
        (r"(?<!\w)yang(?!\w)", "sing", "jawa_ngoko", "sedang"),
        (r"(?<!\w)sekarang(?!\w)", "saiki", "jawa_ngoko", "sedang"),
        (r"(?<!\w)besok(?!\w)", "sesuk", "jawa_ngoko", "sedang"),
    ],
}

# Penambahan kata perintah hanya dipakai ketika frasa pengganti tidak ditemukan.
KATA_PERINTAH = re.compile(
    r"^(?:bisukan|putar|putarkan|mainkan|nyalakan|matikan|atur|ubah|hapus|cek|"
    r"tampilkan|kirim|pesankan|buat|ingatkan|tambahkan|batalkan|beritahu|"
    r"bacakan|nonaktifkan|aktifkan|hentikan|setel|buka|tutup|pasang|periksa|cari)\b", re.I
)


def pola_kata(teks):
    return re.compile(r"(?<!\w)" + re.escape(teks) + r"(?!\w)", re.I)


def teks_gt(val):
    if isinstance(val, dict):
        for v in val.values():
            yield from teks_gt(v)
    elif isinstance(val, (list, tuple)):
        for v in val:
            yield from teks_gt(v)
    elif isinstance(val, str) and val.strip():
        yield val.strip()


def rentang_dilindungi(row):
    q = row["question"]
    result = []
    for value in set(teks_gt(row["gt"])):
        if value.casefold() in {"now", "current", "none", "null"}:
            continue
        for match in pola_kata(value).finditer(q):
            result.append((match.start(), match.end()))
    return result


def terlindungi(start, end, protected):
    return any(start < b and end > a for a, b in protected)


def penggantian_kamus(row, category):
    q = row["question"]
    protected = rentang_dilindungi(row)
    for expr, new, subtype, confidence in POLA[category]:
        m = re.search(expr, q, re.I)
        if m and not terlindungi(m.start(), m.end(), protected):
            text_new = q[:m.start()] + new + q[m.end():]
            if text_new != q:
                return (text_new, q[m.start():m.end()], new, subtype, confidence)
    return None


def variasi_bahasa(row, category):
    found = penggantian_kamus(row, category)
    if found:
        return found
    q = row["question"]
    if category == "code_switch_id_en" and KATA_PERINTAH.match(q):
        return ("please, " + q, "awalan perintah", "please,", "penanda_perintah", "sedang")
    if category == "code_switch_id_jawa" and KATA_PERINTAH.match(q):
        return ("tulung, " + q, "awalan perintah", "tulung,", "penanda_perintah_ngoko", "perlu_penutur")
    return None

print("Kategori berbasis kamus:", list(POLA))

Kategori berbasis kamus: ['slang', 'singkatan', 'code_switch_id_en', 'code_switch_id_jawa']


## **5. Typo Rule-Based**

---

Typo dibuat dengan menukar dua huruf bersebelahan pada satu kata yang cukup panjang. Pilihan kata ditentukan oleh seed tetap. Nama entitas yang muncul dalam nilai GT tidak digunakan sebagai target perubahan. Hasilnya adalah satu variasi salah ketik, bukan perubahan makna yang disengaja.

In [4]:
KATA_LARANGAN_TYPO = {"bisa", "bias", "masa", "mas", "satu", "setu", "saat", "taat", "steel"}


def buat_typo(row):
    q = row["question"]
    protected = rentang_dilindungi(row)
    candidates = []
    for match in re.finditer(r"\b[a-zA-Z]{5,}\b", q):
        if terlindungi(match.start(), match.end(), protected):
            continue
        word = match.group()
        if word.lower() in KATA_LARANGAN_TYPO:
            continue
        if word[0].isupper() and match.start() > 0:
            continue
        idx = next((k for k in range(1, len(word)-1) if word[k] != word[k+1]), None)
        if idx is None:
            continue
        new = word[:idx] + word[idx+1] + word[idx] + word[idx+2:]
        if new.lower() == word.lower() or new.lower() in KATA_LARANGAN_TYPO:
            continue
        candidates.append((match, word, new))
    if not candidates:
        return None
    key = int(hashlib.sha256((str(SEED) + row["row_key"]).encode()).hexdigest()[:12], 16)
    match, old, new = candidates[key % len(candidates)]
    new_q = q[:match.start()] + new + q[match.end():]
    return (new_q, old, new, "transposisi_1x", "tinggi")


def buat_varian(row, category):
    if category == "typo":
        return buat_typo(row)
    return variasi_bahasa(row, category)

assert len(KATEGORI) == 5
print("Aturan typo: satu transposisi dua karakter bersebelahan")

Aturan typo: satu transposisi dua karakter bersebelahan


## **6. Pembentukan Dataset Kandidat**

---

Setiap instruksi induk dapat menghasilkan paling banyak lima variasi. Nilai GT dan informasi grup disalin tanpa perubahan. Hanya satu varian dibuat per kategori dan per induk, sehingga hasilnya dapat dibandingkan per instruksi asal.

In [5]:
hasil = []
tidak_terbentuk = []
induk_info = []

for row in induk:
    func = next(iter(row["gt"]))
    gt_json = json.dumps(row["gt"], sort_keys=True, ensure_ascii=False)
    induk_info.append({
        "row_key_induk": row["row_key"],
        "group_id": row["group_id"],
        "instruksi_id_asli": row["question"],
        "fungsi_gt": func,
        "gt_json": gt_json,
        "status_formal": "BELUM_VALIDASI_MANUAL",
        "status_gt": "BELUM_VALIDASI_MANUAL",
    })
    for cat in KATEGORI:
        variant = buat_varian(row, cat)
        if variant is None:
            tidak_terbentuk.append({"row_key_induk": row["row_key"], "kategori": cat, "alasan": "tidak ada aturan sesuai"})
            continue
        instruksi, asal, ganti, subtipe, confidence = variant
        hasil.append({
            "variant_id": row["row_key"] + "::" + cat,
            "row_key_induk": row["row_key"],
            "source_record_id": row["source_record_id"],
            "group_id": row["group_id"],
            "split": row["split"],
            "language_code": row["language_code"],
            "kategori": cat,
            "subtipe": subtipe,
            "tingkat_review": confidence,
            "instruksi_id_asli": row["question"],
            "instruksi_variasi": instruksi,
            "aturan_asal": asal,
            "aturan_pengganti": ganti,
            "fungsi_gt": func,
            "gt_json": gt_json,
            "status_validasi": "BELUM_VALIDASI_MANUAL",
        })

print("Induk:", len(induk_info))
print("Total kandidat variasi:", len(hasil))
print("Jumlah per kategori:")
for cat in KATEGORI:
    print(" ", cat, sum(x["kategori"] == cat for x in hasil))

Induk: 359
Total kandidat variasi: 931
Jumlah per kategori:
  slang 139
  singkatan 187
  typo 304
  code_switch_id_en 130
  code_switch_id_jawa 171


## **7. Pemeriksaan Konsistensi**

---

Pemeriksaan memastikan fungsi, argumen GT, instruksi asal, `group_id`, dan split tetap sesuai dengan dataset sumber. Pemeriksaan ini hanya bersifat struktural; kewajaran dan kesetaraan makna harus ditinjau terpisah.

In [6]:
by_row_key = {r["row_key"]: r for r in data}
assert len(hasil) == len(set(r["variant_id"] for r in hasil)), "ID varian duplikat"
assert all(r["instruksi_id_asli"] != r["instruksi_variasi"] for r in hasil)
assert len(induk_info) == len({r["group_id"] for r in induk_info}), "group induk duplikat"

check_rows = []
for item in hasil:
    origin = by_row_key[item["row_key_induk"]]
    check_rows.append({
        "variant_id": item["variant_id"],
        "gt_sama": item["gt_json"] == json.dumps(origin["gt"], sort_keys=True, ensure_ascii=False),
        "fungsi_sama": item["fungsi_gt"] == next(iter(origin["gt"])),
        "group_id_sama": item["group_id"] == origin["group_id"],
        "split_sama": item["split"] == origin["split"] == "development",
        "teks_asal_sama": item["instruksi_id_asli"] == origin["question"],
    })
assert all(all(v for k, v in row.items() if k != "variant_id") for row in check_rows)
print("Pemeriksaan sumber, GT, fungsi, group_id, dan split: PASS")
print("Validasi kesetaraan makna: BELUM")

Pemeriksaan sumber, GT, fungsi, group_id, dan split: PASS
Validasi kesetaraan makna: BELUM


## **8. Sampel Audit per Kategori**

---

Target audit adalah maksimal 100 variasi dari setiap kategori. Pemilihan memakai seed tetap dan satu `group_id` per kategori agar instruksi yang berkaitan tidak mendominasi sampel. Kategori dengan kurang dari 100 kandidat dicatat sebagai kekurangan cakupan, bukan diisi dengan variasi buatan yang dipaksakan.

Peninjauan dibagi menjadi formalitas induk, kesamaan makna, kestabilan entitas/waktu, kewajaran variasi, dan kecocokan GT.

In [7]:
rng = random.Random(SEED)
sampel = []
ringkasan = []
for cat in KATEGORI:
    items = [x for x in hasil if x["kategori"] == cat]
    rng.shuffle(items)
    seen = set()
    picked = []
    for item in items:
        if item["group_id"] in seen:
            continue
        seen.add(item["group_id"])
        picked.append(item)
        if len(picked) >= TARGET_AUDIT_PER_KATEGORI:
            break
    for item in picked:
        sampel.append({
            **item,
            "induk_formal": "BELUM_DIREVIEW",
            "makna_tetap": "BELUM_DIREVIEW",
            "entitas_waktu_tetap": "BELUM_DIREVIEW",
            "fungsi_gt_masih_valid": "BELUM_DIREVIEW",
            "bahasa_wajar": "BELUM_DIREVIEW",
            "keputusan_manual": "BELUM_DIREVIEW",
            "alasan": "",
        })
    ringkasan.append({
        "kategori": cat,
        "jumlah_kandidat": len(items),
        "jumlah_sampel_audit": len(picked),
        "target_audit": TARGET_AUDIT_PER_KATEGORI,
        "kekurangan_sampel": max(0, TARGET_AUDIT_PER_KATEGORI - len(picked)),
        "status": "CUKUP_KANDIDAT" if len(picked) >= TARGET_AUDIT_PER_KATEGORI else "KURANG_KANDIDAT",
    })

for r in ringkasan:
    print(f"{r['kategori']:<22} kandidat={r['jumlah_kandidat']:<4} sampel={r['jumlah_sampel_audit']:<4} kekurangan={r['kekurangan_sampel']}")
print("Total sampel audit:", len(sampel))

slang                  kandidat=139  sampel=100  kekurangan=0
singkatan              kandidat=187  sampel=100  kekurangan=0
typo                   kandidat=304  sampel=100  kekurangan=0
code_switch_id_en      kandidat=130  sampel=100  kekurangan=0
code_switch_id_jawa    kandidat=171  sampel=100  kekurangan=0
Total sampel audit: 500


## **9. Penyimpanan Hasil**

---

Semua keluaran disimpan di folder baru `development_v2`. Folder pilot 40 dan hasil evaluasi baseline tidak ditimpa. Sampel audit belum menjadi dataset tervalidasi sampai keputusan manual selesai.

In [8]:
def simpan_csv(rows, filename):
    path = OUTPUT / filename
    if not rows:
        raise ValueError("Tidak ada data untuk file: " + filename)
    headers = list(rows[0].keys())
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=headers)
        writer.writeheader()
        writer.writerows(rows)
    return path

files = [
    simpan_csv(induk_info, "kandidat_induk_formal_v2.csv"),
    simpan_csv(hasil, "variasi_bahasa_kandidat_v2.csv"),
    simpan_csv(sampel, "sampel_validasi_5_kategori_v2.csv"),
    simpan_csv(check_rows, "pemeriksaan_konsistensi_v2.csv"),
    simpan_csv(ringkasan, "ringkasan_kategori_v2.csv"),
]
if tidak_terbentuk:
    files.append(simpan_csv(tidak_terbentuk, "kategori_tidak_terbentuk_v2.csv"))

meta = {
    "run": "variasi_bahasa_development_v2",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "input_sha256": hashlib.sha256(INPUT.read_bytes()).hexdigest(),
    "seed": SEED,
    "input_rows": len(data),
    "parents": len(induk),
    "variants": len(hasil),
    "audit_rows": len(sampel),
    "counts": {x["kategori"]: x["jumlah_kandidat"] for x in ringkasan},
    "audit_shortfall": {x["kategori"]: x["kekurangan_sampel"] for x in ringkasan},
    "split": "development",
    "test_used": False,
    "gt_modified": False,
    "model_generator_used": False,
    "semantic_validation": "pending_human_review",
    "ready_for_final_model_evaluation": False,
    "audit_note": "Target maksimal 100 per kategori; kurangnya kandidat ditampilkan terbuka.",
}
meta_file = OUTPUT / "metadata_development_v2.json"
meta_file.write_text(json.dumps(meta, indent=2, ensure_ascii=False), encoding="utf-8")
files.append(meta_file)
print("Output yang tersimpan:")
for path in files:
    print("-", path.name)

Output yang tersimpan:
- kandidat_induk_formal_v2.csv
- variasi_bahasa_kandidat_v2.csv
- sampel_validasi_5_kategori_v2.csv
- pemeriksaan_konsistensi_v2.csv
- ringkasan_kategori_v2.csv
- kategori_tidak_terbentuk_v2.csv
- metadata_development_v2.json


## **10. Kesimpulan Sementara**

---

Dataset development versi 2 merupakan **calon variasi bahasa lokal**, bukan data yang sudah lolos validasi semantik. Pemeriksaan GT dan pemilihan fungsi masih menggunakan anotasi asli. Hasil ini belum digunakan untuk menyimpulkan ketahanan model terhadap bahasa informal. Validasi independen tetap dibutuhkan sebelum eksperimen berikutnya.